### Data Preparation and Ingestion

##### Focus: Humanitarian supply chain realities in 2025, including crisis impacts (conflict, floods, funding cuts, access blocks) in key countries.  

#### Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

#### Realistic Parameters
- Countries: Focus on high-need humanitarian contexts.
- Item types & codes: Based on common humanitarian relief items (food, medical, NFI, nutrition).
- Biases: Lower stock/high expiry in crisis zones; compliance issues in restricted areas.

#### Set Up Parameters

In [2]:
ref_date = datetime(2025, 6, 1)

countries = ['Ethiopia', 'Yemen', 'Bangladesh', 'South Sudan', 'Afghanistan']
country_weights = [0.30, 0.25, 0.15, 0.20, 0.10] 

item_categories = {
    'Food Rations': ['FR-001', 'FR-002', 'FR-003'],          # e.g., CSB+, Plumpy'Sup, rice/beans
    'Nutrition Supplements': ['NS-001', 'NS-002'],           # RUTF, RUSF for SAM
    'Medical Supplies': ['MS-001', 'MS-002', 'MS-003'],      # Antibiotics, IV fluids, malaria kits
    'Hygiene Kits': ['HK-001', 'HK-002'],                     # Soap, buckets, menstrual hygiene
    'NFI - Shelter': ['NFI-S01', 'NFI-S02'],                 # Tarps, blankets, sleeping mats
    'NFI - WASH': ['NFI-W01', 'NFI-W02']                     # Jerry cans, latrine kits
}

n_rows = 1500

#### Generate Core Dataset  

In [3]:
selected_countries = np.random.choice(countries, size=n_rows, p=country_weights)

data = {
    'warehouse_id': [f'WH-{random.randint(1000, 9999):04d}-{c[:3].upper()}' for c in selected_countries],
    'country': selected_countries,
    'item_code': [],
    'item_type': [],
    'batch_number': [f'BATCH-{random.randint(10000, 99999)}' for _ in range(n_rows)],  # WIM batch/lot tracking
    'stock_level_current': [],
    'stock_level_initial': np.random.randint(500, 5000, n_rows),  # Starting quantity on receipt
    'unit': np.random.choice(['kg', 'pcs', 'kits', 'bottles'], n_rows),
    'last_received_date': [],
    'last_distributed_date': [],
    'expiry_date': [],
    'condition_flag': [],  # WIM quality flag
    'donor_compliance_status': [],  # Audit/donor flag
    'waybill_reference': [f'WB-{random.randint(100000, 999999)}' for _ in range(n_rows)]  # Logistics tracking
}

for i in range(n_rows):
    country = selected_countries[i]
    
    # Item type biased by country crisis
    if country in ['Ethiopia', 'South Sudan', 'Bangladesh']:
        item_type = random.choices(['Food Rations', 'Nutrition Supplements', 'NFI - Shelter'], weights=[0.45, 0.35, 0.20])[0]
    elif country == 'Yemen':
        item_type = random.choices(['Medical Supplies', 'Hygiene Kits', 'Food Rations'], weights=[0.40, 0.35, 0.25])[0]
    else:  # Afghanistan
        item_type = random.choices(['Nutrition Supplements', 'Medical Supplies', 'Hygiene Kits'], weights=[0.40, 0.35, 0.25])[0]
    
    data['item_type'].append(item_type)
    data['item_code'].append(random.choice(item_categories[item_type]))
    
    # Stock realism: often low due to high demand/funding cuts
    initial = data['stock_level_initial'][i]
    depletion_factor = random.uniform(0.60, 0.95) if random.random() < 0.7 else random.uniform(0.20, 0.60)  # High depletion common
    current_stock = max(0, int(initial * depletion_factor))
    data['stock_level_current'].append(current_stock)
    
    # Dates: realistic humanitarian delays
    received_offset = random.randint(30, 540)   # Received 1-18 months ago
    distributed_offset = random.randint(0, received_offset)
    expiry_offset = random.randint(90, 730)     # Expiry 3-24 months from receipt
    
    received_date = ref_date - timedelta(days=received_offset)
    expiry_date = received_date + timedelta(days=expiry_offset)
    
    data['last_received_date'].append(received_date)
    data['last_distributed_date'].append(received_date + timedelta(days=distributed_offset) if current_stock < initial else None)
    data['expiry_date'].append(expiry_date)
    
    # Condition & compliance realism
    if expiry_date < ref_date:
        condition = random.choices(['Expired', 'Damaged/Flood', 'Good'], weights=[0.55, 0.15, 0.30])[0]
    elif (expiry_date - ref_date).days < 90:
        condition = random.choices(['Near Expiry', 'Good'], weights=[0.60, 0.40])[0]
    else:
        condition = 'Good'
    
    data['condition_flag'].append(condition)
    
    # Compliance: higher issues in restricted/conflict areas
    if country in ['Yemen', 'Afghanistan', 'South Sudan']:
        compliance = random.choices(['Compliant', 'Pending Audit', 'Non-Compliant - Access Blocked', 'Non-Compliant - Documentation'], weights=[0.40, 0.30, 0.20, 0.10])[0]
    else:
        compliance = random.choices(['Compliant', 'Pending Audit'], weights=[0.70, 0.30])[0]
    data['donor_compliance_status'].append(compliance)

#### Create DataFrame and Add Derived Realism

In [4]:
df_inventory = pd.DataFrame(data)

# Add expiry flag (key WIM KPI)
df_inventory['is_expired'] = df_inventory['expiry_date'] < ref_date
df_inventory['days_to_expiry'] = (df_inventory['expiry_date'] - ref_date).dt.days

# Sort for realism (by country/warehouse)
df_inventory = df_inventory.sort_values(['country', 'warehouse_id'])

print("Dataset Info:")
print(df_inventory.info())
print("\nSample Rows:")
print(df_inventory.head(10))
print("\nExpiry Summary:")
print(df_inventory['is_expired'].value_counts(normalize=True))

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1500 entries, 1395 to 184
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   warehouse_id             1500 non-null   object        
 1   country                  1500 non-null   object        
 2   item_code                1500 non-null   object        
 3   item_type                1500 non-null   object        
 4   batch_number             1500 non-null   object        
 5   stock_level_current      1500 non-null   int64         
 6   stock_level_initial      1500 non-null   int32         
 7   unit                     1500 non-null   object        
 8   last_received_date       1500 non-null   datetime64[ns]
 9   last_distributed_date    1500 non-null   datetime64[ns]
 10  expiry_date              1500 non-null   datetime64[ns]
 11  condition_flag           1500 non-null   object        
 12  donor_compliance_status

#### Export Realistic Dataset
Save as CSV for cleaning and analytics in subsequent phases.  
This file simulates what a global WIM/ERP system export might look like from CARE country offices.

In [5]:
df_inventory.to_csv('inventory_data.csv', index=False)
print("Exported realistic inventory_data.csv with", len(df_inventory), "rows.")

Exported realistic inventory_data.csv with 1500 rows.


In [7]:
df_inventory.tail()

,warehouse_id,country,item_code,item_type,batch_number,stock_level_current,stock_level_initial,unit,last_received_date,last_distributed_date,expiry_date,condition_flag,donor_compliance_status,waybill_reference,is_expired,days_to_expiry
232,WH-9882-YEM,Yemen,MS-003,Medical Supplies,BATCH-13809,3184,3359,kg,2024-06-01,2025-03-30,2025-07-03,Near Expiry,Non-Compliant - Access Blocked,WB-459608,False,32
851,WH-9908-YEM,Yemen,HK-001,Hygiene Kits,BATCH-82986,3818,4235,kg,2024-12-25,2025-03-04,2026-10-01,Good,Non-Compliant - Documentation,WB-386165,False,487
1108,WH-9937-YEM,Yemen,MS-003,Medical Supplies,BATCH-71209,1770,3275,pcs,2024-01-19,2024-09-04,2025-01-14,Good,Compliant,WB-690176,True,-138
1439,WH-9940-YEM,Yemen,FR-003,Food Rations,BATCH-65637,632,1439,kg,2024-10-24,2025-01-12,2026-07-26,Good,Compliant,WB-247468,False,420
184,WH-9943-YEM,Yemen,HK-001,Hygiene Kits,BATCH-49053,1134,1453,kg,2024-01-19,2025-01-22,2025-11-20,Good,Pending Audit,WB-768915,False,172
